# Collecte des événements culturels — Open Agenda (Rennes)

**Objectif :** récupérer tous les événements culturels à Rennes via l'API publique OpenDataSoft,  
filtrer sur moins d'un an, et sauvegarder le résultat en CSV dans `data/raw/`.

**Source :** https://public.opendatasoft.com/explore/dataset/evenements-publics-openagenda  
**Pas de clé API requise** — données publiques.

In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone

## 1. Exploration de l'API

Avant de tout récupérer, on fait un premier appel avec `limit=1` pour examiner la structure d'un enregistrement.  
C'est essentiel pour savoir quels champs existent et lesquels sont utiles pour notre RAG.

In [2]:
BASE_URL = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/evenements-publics-openagenda/records"

response = requests.get(BASE_URL, params={
    "where": 'location_city="Rennes"',
    "limit": 1
})

response.raise_for_status()
data = response.json()

print(f"Total disponible : {data['total_count']}")

import json
print(json.dumps(data['results'][0], indent=2, ensure_ascii=False))

Total disponible : 17046
{
  "uid": "43470055",
  "slug": "les-mots-en-toute-liberte-par-la-balade-des-livres-3394679",
  "canonicalurl": "https://openagenda.com/rennes-metropole/events/les-mots-en-toute-liberte-par-la-balade-des-livres-3394679",
  "title_fr": "Les mots en toute liberté par La Balade des Livres",
  "description_fr": "Découvrir le plaisir de la lecture à haute voix et le partager.",
  "longdescription_fr": "<hr />\n<p>Découvrir le plaisir de la lecture à haute voix et le partager.</p>\n<p></p>\n<p>Accompagnement par une artiste-auteure professionnelle</p>\n<p></p>\n<p>Pour des enfants, à partir de 8 ans et leurs parents, curieux de découvrir des textes, des contes, des albums... des univers d'<a href=\"http://auteur.es\">auteur.es,</a></p>\n<p></p>\n<p>désireux d'explorer leurs capacités vocales, corporelles... et de vivre ensemble un agréable moment.</p>\n<p></p>\n<p>Cet événement s'inscrit dans le programme d'un Été culturel 2023. Il est soutenu par le Ministère de la

## 2. Analyse de la structure

Pour le RAG, on distingue deux types de champs :

**Contenu indexé** (embedé et recherché) :
- `longdescription_fr`

**Métadonnées** (enrichissent la réponse générée) :
- `uid`, `title_fr` — identification
- `firstdate_begin`, `lastdate_end` — pour les questions temporelles
- `location_name`, `location_address` — pour les questions de lieu
- `conditions_fr` — pour les questions de tarif

## 3. Filtre temporel

On ajoute un filtre `lastdate_end >= aujourd'hui - 1 an` pour ne garder que les événements récents.  
On vérifie d'abord combien d'événements passent ce filtre avant de tout télécharger.

In [3]:
date_limit = (datetime.now(timezone.utc) - timedelta(days=365)).strftime("%Y-%m-%dT%H:%M:%SZ")

response = requests.get(BASE_URL, params={
    "where": f'location_city="Rennes" AND lastdate_end >= "{date_limit}"',
    "limit": 1
})

response.raise_for_status()
print(f"Événements dans la fenêtre d'un an : {response.json()['total_count']}")

Événements dans la fenêtre d'un an : 5745


## 4. Collecte complète avec pagination

L'API limite à 100 résultats par requête. On boucle en incrémentant `offset` jusqu'à avoir tout récupéré.

In [4]:
FIELDS = "uid,title_fr,longdescription_fr,conditions_fr,firstdate_begin,lastdate_end,location_name,location_address"

all_records = []
offset = 0
total = 5745

while offset < total:
    response = requests.get(BASE_URL, params={
        "where": f'location_city="Rennes" AND lastdate_end >= "{date_limit}"',
        "select": FIELDS,
        "limit": 100,
        "offset": offset
    })
    response.raise_for_status()
    records = response.json()["results"]
    all_records.extend(records)
    offset += 100
    print(f"{len(all_records)}/{total}", end="\r")

print(f"\nCollecte terminée : {len(all_records)} événements")

5745/5745
Collecte terminée : 5745 événements


## 5. Sauvegarde en CSV

In [5]:
df = pd.DataFrame(all_records)

print(df.shape)
print(df.dtypes)
df.head(3)

(5745, 8)
uid                   object
title_fr              object
longdescription_fr    object
conditions_fr         object
firstdate_begin       object
lastdate_end          object
location_name         object
location_address      object
dtype: object


,uid,title_fr,longdescription_fr,conditions_fr,firstdate_begin,lastdate_end,location_name,location_address
0,10733637,Boostez Votre Français et Rejoignez le Monde d...,<p>Vous avez besoin d'améliorer votre niveau d...,None,2025-10-14T07:00:00+00:00,2025-10-14T10:00:00+00:00,Agence RENNES EST,35000 Rennes
1,71218018,Jeudis de la formation- Venez rencontrer le CLPS,<p>Le CLPS est un organisme de formation génér...,None,2025-08-21T08:00:00+00:00,2025-08-21T09:00:00+00:00,Agence RENNES SUD,35200 Rennes
2,76724559,"**Webinaire "" travailler pour un moulin, pourq...",<p>Oui les moulins pour fabriquer de la farine...,None,2025-11-03T10:00:00+00:00,2025-11-03T10:30:00+00:00,Rennes - BRETAGNE,35000 Rennes


In [6]:
df.to_csv("../data/raw/events.csv", index=False, encoding="utf-8")
print("Sauvegardé dans data/raw/events.csv")

Sauvegardé dans data/raw/events.csv
